In [2]:
##alles was hier kommt sind nur rundungsfunktionen und eine voreinstellung für die Achsenbeschriftung



using Measurements
import Measurements: value, uncertainty
using Statistics



using Random, Statistics

function first_significant_digit(x::Real)
    x == 0 && return 0  # Sonderfall: Wenn x genau 0 ist, gibt es keine signifikante Stelle, also gib 0 zurück.

    absx = abs(x)  # Der Betrag der Zahl wird genommen, damit negative Vorzeichen ignoriert werden.

    # Berechne die Zehnerpotenz, in der sich die erste signifikante Stelle befindet.
    # log10(absx) gibt den dekadischen Logarithmus. floor(Int, ...) rundet nach unten auf ganze Zahl.
    exponent = floor(Int, log10(absx))

    # Teile absx durch 10^exponent, um die Zahl in den Bereich [1,10) zu bringen
    # Beispiel: aus 456 → 4.56, aus 0.00456 → 4.56
    significand = absx / 10.0^exponent

    # Jetzt holen wir die ganzzahlige Ziffer vor dem Komma, das ist die erste signifikante Ziffer
    return floor(Int, significand)
end


"""
    round_measurement(m::Measurement; sigdigits_error::Int=1)

Rundet erst die Unsicherheit von `m` auf `sigdigits_error` signifikante Stellen
und rundet dann den Messwert auf dieselbe Genauigkeit (Dezimalstellen),
liefert einen neuen `Measurement`. Sigdigits_error wird automatisch auf den richtigen wert gesetzt durch if bedingung 
wenn first sig digit =1 oder 2, wird eine weitere stelle hinzugenommen.

input: ein measurement, output: ein measurement. Kann punktweise angewendet werden
"""
function round_measurement(m::Measurement)
    # 1) Unsicherheit extrahieren und auf sigdigits_error sig. Stellen runden
    u = uncertainty(m)
    if first_significant_digit(u)<3
        sigdigits_error=2
    else
        sigdigits_error=1
    end
    #println(sigdigits_error)

    u_r = round(u; sigdigits=sigdigits_error)

    
    if u_r==0 #wenn gauß versagt, normalverteilung und stw
        

    # Ziehe N Stichproben aus einer Normalverteilung und berechne KINETISCHE ENERGIE!!!
    p_samples = randn(100_000) .* 0.05  # p ~ N(0, σp=0.05) kein plan warum, aber die 0.05 sind auf jkeden fall der fehler vorher

    # Berechne kinetische Energie für jede Probe
    E_samples = p_samples.^2 ./ (2)

    # Mittelwert und Standardabweichung der Energie
    E_mean = mean(E_samples)
    E_std = std(E_samples)
    
    u_r = round(E_std; sigdigits=sigdigits_error)
        #println(u_r)
    end
    
    
    
    
    
    # 2) Anzahl Dezimalstellen bestimmen:
    #    Wenn u_r = x * 10^e  (mit 1 ≤ x < 10), dann ist e = floor(log10(u_r))
    #    und wir benötigen -e  Dezimalstellen (für e ≤ 0)
    e = floor(Int, log10(u_r))
       # println(e)
    if first_significant_digit(u)<3
        dec = max(0, -e) +1
    else
        dec = max(0, -e)
    end
    #dec = max(0, -e)
    #println(dec, log(12,u_r))

    # 3) Wert runden und neuen Measurement erstellen
    v_r = round(value(m); digits=dec)
    return measurement(v_r, u_r)
end

# Beispiel
m = measurement(0.023456, 0.0120236789)
println("Original: ", value(m))                  # 1.23456 ± 0.06789
m2 = round_measurement(m)
println("Gerundet: ", value(m2))  # z.B. 1.23(7) → ±0.07, Wert 1.23




Original: 0.023456
Gerundet: 0.023


In [3]:
#=diese funktion generiert die reflexe für sc, bcc und fcc, wobei hmax
einfach eine obere grenze für werte von h,k,l ist, die wir uns 
anschauen und nth die anzahl der reflexe ist, die man haben will. Diese Code liefert Literaturwerte (https://web.mit.edu/turzaak/Public/3.091/3091_test2_notes.pdf) =#


function generate_reflections(hmax,nth)
    sc  = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
    bcc = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
    fcc = Dict{Int, Vector{Tuple{Int,Int,Int}}}()

    for h in 0:hmax, k in 0:hmax, l in 0:hmax

        N = h^2 + k^2 + l^2
        hkl = (h,k,l)

        # --- SC: alle erlaubten Reflexe ---
        if length(sc) == nth
            if N < maximum(keys(sc))
                delete!(sc, maximum(keys(sc)))
            end
        end 

        if length(fcc) == nth
            if N < maximum(keys(fcc))
                delete!(fcc, maximum(keys(fcc)))
            end
        end 

        if length(bcc) == nth
            if N < maximum(keys(bcc))
                delete!(bcc, maximum(keys(bcc)))
            end
        end 


        if length(sc) < nth
            push!(get!(sc, N, Vector{Tuple{Int,Int,Int}}()), hkl)
        end

        # --- BCC: h+k+l gerade ---
        if length(bcc) < nth
            if (h + k + l) % 2 == 0
                push!(get!(bcc, N, Vector{Tuple{Int,Int,Int}}()), hkl) 
            end
        end

        # --- FCC: alle h,k,l entweder alle gerade oder alle ungerade ---
        if length(fcc) < nth
            if (h % 2 == k % 2 == l % 2)
                push!(get!(fcc, N, Vector{Tuple{Int,Int,Int}}()), hkl)
            end
        end
    end
    sc_sorted = [(N, sc[N]) for N in sort(collect(keys(sc)))]
    bcc_sorted = [(N, bcc[N]) for N in sort(collect(keys(bcc)))]
    fcc_sorted = [(N, fcc[N]) for N in sort(collect(keys(fcc)))]
    return sc_sorted, bcc_sorted, fcc_sorted
end


generate_reflections (generic function with 1 method)

In [201]:
#=berechnet die theoretisch erwarteten winkelzwischen den ebenen des fcc gitters mit der 111 ebene =#
function anglestheo(n)
    rfl=generate_reflections(n,n)[3] #erwartete reflexe bei fcc
    α = zeros(n)
    for i in 1:n
        if rfl[i][2][1] == (0,0,0) #der durchlaufende strahl hat keinen reflex
        α[i] = 180
        else
        α[i] = acos((rfl[i][2][1][1]+rfl[i][2][1][2]+rfl[i][2][1][3])/(sqrt(3)* sqrt((rfl[i][2][1][1]^2+rfl[i][2][1][2]^2+rfl[i][2][1][3]^2)))-0.00000000002)/pi*180
        end
    end
return round_measurement.(measurement.(α,0.001)),rfl
end
#also ich berechne hier einfach nur die winkel zwischen ebenen mithilfe von millerindizes. das entspricht den winkeln ihrer normalen also ihrer rez. gittervektoren
# wegen kubischen gitter kann ich einfaches eukl. SP benutzen
#
#gleich schaue ich mir dann die gemessenen reflexe an und kriege damit die richtung von k´(ich sage k, k´haben selben betrag). aus k-k´kann ich die richtung von g berechnen.
#mit diesem g kann ich dann den winkel zu k_111 berechnen, wobei k_111 parallel zu 1 1 1 ist. Bei dem Ausgerichtetem 
#Laue bild entspricht das einfach unserem einfallenden k. Damit kann man dann die die reflexe zuordnen. 

anglestheo (generic function with 1 method)

In [202]:
using LinearAlgebra

In [203]:
#dieser code hier wäre die rechnung mit einfachen Polarkoordinaten. Ich weiß nicht, warum in der versuchsbeschreibung die umrechnug
#der greningerkarte von kartesischen in "Polarkoordinaten entlang der y achse" und nicht normale polarkoordinaten ist, aber habe es dann auch so gemacht.

#=
function anglesprax(th, ph,th111,ph111)
    th = th .* pi/180
    ph = ph .* pi/180
    kk(x,y)=[(sin(x)*cos(y))';
     (sin(x).*sin(y))';
      (cos(x)')]
    #println(kk.(th,ph))
    kkvec=kk.(th,ph)
    kvec = kk(th111*pi/180, ph111*pi/180)
    g = zeros(3,length(th))
    #println(g)
    h= zeros(length(th))
    kg=(kvec-[0,0,-1])/norm(kvec-[0,0,-1])
    for i in 1:length(th)
        #println(kvec)
        g[:, i] =  kkvec[i]-[0,0,-1]
        #println(g[:, i])
        g[:, i]=g[:, i]/norm(g[:, i])
        h[i] = acos(kg' *g[:, i])/pi*180
    end
    return h
end =#

In [204]:
function anglespraxy(th, ph,th111,ph111)
    th = measurement.(th,1) .* pi/180   #winkel in radian
    ph = measurement.(ph,1) .* pi/180
    println(uncertainty.(th))
    th111 = measurement.(th111,1) .* pi/180
    ph111 = measurement.(ph111,1) .* pi/180
    kk(x,y)=[(cos(x)*sin(y)),      #umrechnung der greningerkarte in normierte kart. koord.
     (cos(x)*cos(y)),
      (sin(x))]
    #println(kk.(th,ph))
    vec111 = kk(th111, ph111) #umrechnung der 111 ebene in kartesische koordinaten
    g = zeros(3,length(th))
    g = measurement.(g, 0)  #das werden später die reziproken gittervektoren der der reflexe, also \Delta k
    h= zeros(length(th))
    h= measurement.(h, 0) #das werden später die winkel
    for i in 1:length(th)
        g[:, i] =  kk(th[i], ph[i]) 
        h[i] = acos(dot( vec111, g[:, i])-0.0000000000000003)/pi*180  # winkel zwischen (1,1,1) rez. gittervektor und i-tem reflex- rez. gittervektor 
        #und umrechnung zurück in grad
        if round(abs(value.(h[i])), digits=5) == 0
            h[i] = measurement(0,0)
        end
    end
    h= round_measurement.(h)
    println(uncertainty.(h))
    return h
end

anglespraxy (generic function with 1 method)

In [205]:
(2.998*(10^(5)))^2 *7.7 *10^(-10) 

69.2076308

In [206]:
#unorientierte phi
ph = -1 .*[5.5, 2, -8, 10, 6.5, 2, -4, -13, -25, 20.5, 8, -6.5, -5.5, -16, -18]
th = [28, 26, 26, 18, 17.5, 16.5, 16, 15, 13.5, 8, 6.5, 7, 6, 6, 1.5]


15-element Vector{Float64}:
 28.0
 26.0
 26.0
 18.0
 17.5
 16.5
 16.0
 15.0
 13.5
  8.0
  6.5
  7.0
  6.0
  6.0
  1.5

In [207]:
#orientierte 
pho = [3.0, 11.0, 19.0, -16.1, -18.0, -13.1, 8.0, -4.1, -22.0, -3.9, -13.2, 13.3, 2.1]
tho = [20.1, 14.1, 8.0, 12.9, 2.0, 1.9, 10.1, 10.0, 2.2, 8.1, 10.0, 17.8, 16.1]

13-element Vector{Float64}:
 20.1
 14.1
  8.0
 12.9
  2.0
  1.9
 10.1
 10.0
  2.2
  8.1
 10.0
 17.8
 16.1

In [208]:
generate_reflections(14,14) 

([(0, [(0, 0, 0)]), (1, [(0, 0, 1), (0, 1, 0), (1, 0, 0)]), (2, [(0, 1, 1), (1, 0, 1), (1, 1, 0)]), (3, [(1, 1, 1)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (5, [(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]), (6, [(1, 1, 2), (1, 2, 1), (2, 1, 1)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (9, [(0, 0, 3), (0, 3, 0), (1, 2, 2), (2, 1, 2), (2, 2, 1), (3, 0, 0)]), (10, [(0, 1, 3), (0, 3, 1), (1, 0, 3), (1, 3, 0), (3, 0, 1), (3, 1, 0)]), (11, [(1, 1, 3), (1, 3, 1), (3, 1, 1)]), (12, [(2, 2, 2)]), (13, [(0, 2, 3), (0, 3, 2), (2, 0, 3), (2, 3, 0), (3, 0, 2), (3, 2, 0)]), (14, [(3, 2, 1)])], [(0, [(0, 0, 0)]), (2, [(0, 1, 1), (1, 0, 1), (1, 1, 0)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (6, [(1, 1, 2), (1, 2, 1), (2, 1, 1)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (10, [(0, 1, 3), (0, 3, 1), (1, 0, 3), (1, 3, 0), (3, 0, 1), (3, 1, 0)]), (12, [(2, 2, 2)]), (14, [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]), (16, [(0, 0, 4), (0, 4, 0), (4, 0, 0)]), (

In [209]:
anglespraxy(ph, th, 13, 15)

[0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295]
[1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 0.0018, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4]


15-element Vector{Measurement{Float64}}:
 22.6 ± 1.4
 18.6 ± 1.4
 11.9 ± 1.4
 23.2 ± 1.4
 19.7 ± 1.4
 15.1 ± 1.4
  9.1 ± 1.4
  0.0 ± 0.0018
 12.1 ± 1.4
 34.2 ± 1.4
 22.6 ± 1.4
 10.2 ± 1.4
 11.6 ± 1.4
  9.2 ± 1.4
 13.9 ± 1.4

In [210]:
#gespiegelt

delta = [-10, 8, 12, 19, 11, -13, 2, 9, 20, -5, 7, -14, 4]

gamma = [-12, -6, -19, -4, -28, -1, -12, -2, -8, -6, -13, -18, -8]


13-element Vector{Int64}:
 -12
  -6
 -19
  -4
 -28
  -1
 -12
  -2
  -8
  -6
 -13
 -18
  -8

In [211]:
gespiegelt= anglespraxy(delta, gamma, 13, 15)

[0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295]
[1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4]


13-element Vector{Measurement{Float64}}:
 35.3 ± 1.4
 21.2 ± 1.4
 33.2 ± 1.4
 19.2 ± 1.4
 42.1 ± 1.4
 30.5 ± 1.4
 28.9 ± 1.4
 17.2 ± 1.4
 23.1 ± 1.4
 27.6 ± 1.4
 28.2 ± 1.4
 42.4 ± 1.4
 24.4 ± 1.4

In [212]:
nicht_gespiegelt= anglespraxy(ph, th, 13, 15)

[0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295, 0.017453292519943295]
[1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 0.0018, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4]


15-element Vector{Measurement{Float64}}:
 22.6 ± 1.4
 18.6 ± 1.4
 11.9 ± 1.4
 23.2 ± 1.4
 19.7 ± 1.4
 15.1 ± 1.4
  9.1 ± 1.4
  0.0 ± 0.0018
 12.1 ± 1.4
 34.2 ± 1.4
 22.6 ± 1.4
 10.2 ± 1.4
 11.6 ± 1.4
  9.2 ± 1.4
 13.9 ± 1.4

In [213]:
angle, order = anglestheo(50)

(Measurement{Float64}[180.0 ± 0.001, 0.0004 ± 0.001, 54.7356 ± 0.001, 35.2644 ± 0.001, 29.4962 ± 0.001, 0.0004 ± 0.001, 54.7356 ± 0.001, 22.0017 ± 0.001, 39.2315 ± 0.001, 19.4712 ± 0.001  …  41.3681 ± 0.001, 32.5125 ± 0.001, 47.4096 ± 0.001, 35.2644 ± 0.001, 40.8303 ± 0.001, 25.2394 ± 0.001, 37.6161 ± 0.001, 33.6443 ± 0.001, 28.5608 ± 0.001, 54.7356 ± 0.001], [(0, [(0, 0, 0)]), (3, [(1, 1, 1)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (11, [(1, 1, 3), (1, 3, 1), (3, 1, 1)]), (12, [(2, 2, 2)]), (16, [(0, 0, 4), (0, 4, 0), (4, 0, 0)]), (19, [(1, 3, 3), (3, 1, 3), (3, 3, 1)]), (20, [(0, 2, 4), (0, 4, 2), (2, 0, 4), (2, 4, 0), (4, 0, 2), (4, 2, 0)]), (24, [(2, 2, 4), (2, 4, 2), (4, 2, 2)])  …  (116, [(0, 4, 10), (0, 10, 4), (4, 0, 10), (4, 6, 8), (4, 8, 6), (4, 10, 0), (6, 4, 8), (6, 8, 4), (8, 4, 6), (8, 6, 4), (10, 0, 4), (10, 4, 0)]), (120, [(2, 4, 10), (2, 10, 4), (4, 2, 10), (4, 10, 2), (10, 2, 4), (10, 4, 2)]), (123, [(1, 1, 11), (1, 11, 1), (5

In [216]:

for i in 1:length(gespiegelt)
    c=1
    candidate=[0.0, "nicht zuordbar"]
    for j in 1:length(angle)
        if abs(gespiegelt[i]-angle[j])<c
            candidate = [angle[j], order[j][2][1]]
            c = abs(gespiegelt[i]-angle[j])
        end
    end
    println(" Reflex $(i): "," gemessen: ", gespiegelt[i], candidate[2], " Winkel: ", value.(candidate[1]))
end
println("

nicht gespiegelt

")
for i in 1:length(nicht_gespiegelt)
    c=1
    candidate=[0.0, "nicht zuordbar"]
    for j in 1:length(angle)
        if abs(nicht_gespiegelt[i]-angle[j])<c
            candidate = [angle[j], order[j][2][1]]
            c = abs(nicht_gespiegelt[i]-angle[j])
        end
    end
    if i != 8
    println(" Reflex $(i): ", " gemessen: ", nicht_gespiegelt[i] , candidate[2], " Winkel: ", value.(candidate[1]))
    else
    println(" Reflex $(i): ", " gemessen: ", 0, " " , candidate[2], " Winkel: ", 0)
    end
end



 Reflex 1:  gemessen: 35.3 ± 1.4(0, 2, 2) Winkel: 35.2644
 Reflex 2:  gemessen: 21.2 ± 1.4(1, 3, 3) Winkel: 22.0017
 Reflex 3:  gemessen: 33.2 ± 1.4(1, 5, 9) Winkel: 33.1525
 Reflex 4:  gemessen: 19.2 ± 1.4(2, 2, 4) Winkel: 19.4712
 Reflex 5:  gemessen: 42.1 ± 1.4(0, 4, 10) Winkel: 41.3681
 Reflex 6:  gemessen: 30.5 ± 1.4(1, 5, 7) Winkel: 29.9264
 Reflex 7:  gemessen: 28.9 ± 1.4(1, 3, 5) Winkel: 28.5608
 Reflex 8:  gemessen: 17.2 ± 1.4nicht zuordbar Winkel: 0.0
 Reflex 9:  gemessen: 23.1 ± 1.4(3, 3, 7) Winkel: 23.5159
 Reflex 10:  gemessen: 27.6 ± 1.4(2, 4, 8) Winkel: 28.1255
 Reflex 11:  gemessen: 28.2 ± 1.4(2, 4, 8) Winkel: 28.1255
 Reflex 12:  gemessen: 42.4 ± 1.4(0, 2, 6) Winkel: 43.0887
 Reflex 13:  gemessen: 24.4 ± 1.4(3, 5, 9) Winkel: 23.7589


nicht gespiegelt


 Reflex 1:  gemessen: 22.6 ± 1.4(2, 4, 6) Winkel: 22.2077
 Reflex 2:  gemessen: 18.6 ± 1.4(2, 2, 4) Winkel: 19.4712
 Reflex 3:  gemessen: 11.9 ± 1.4nicht zuordbar Winkel: 0.0
 Reflex 4:  gemessen: 23.2 ± 1.4(3, 3, 7) Wi